In [1]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────
# Run once per Colab session — installs all pipeline dependencies.
#
# ⚠️  langchain-google-genai MUST be pinned to < 2.0
#     v2+ uses the google-genai SDK which routes to v1beta where
#     text-embedding-004 is unavailable via embedContent → 404 NOT_FOUND.
#     v1.x uses google-generativeai (stable v1 API) — works correctly.
%pip install langchain langchain-community "langchain-google-genai<2.0" langchain-chroma \
             langchain-text-splitters \
             PyMuPDF pdfplumber sentence-transformers chromadb \
             tqdm tenacity python-dotenv -q
print('✅ Dependencies installed')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 67.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 87.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 71.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.9/146.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 598.7/598.7 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━

In [2]:
# ── Cell 2: Setup — Drive (data) + GitHub clone (code) ───────────────────
#
# Why two roots?
#   CODE_ROOT  → cloned from GitHub to Colab's local /content/
#                Makes app/ importable without uploading anything to Drive.
#   DATA_ROOT  → Google Drive folder you already have with PDFs
#                All heavy data (PDFs + chroma_db) stays on Drive.

import os

# ── 2a. Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive # type: ignore
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/smart-learning-assistant'

# Create all required directories on Drive (safe if they already exist)
for sub in ['data/chroma_db',
            'data/raw/1_textbooks',
            'data/raw/2_core_vision',
            'data/raw/3_python_utilities']:
    os.makedirs(f'{DATA_ROOT}/{sub}', exist_ok=True)

print(f'✅ Drive mounted.  DATA_ROOT : {DATA_ROOT}')
print()
print('📂 Before running Cell 5, upload your PDFs to Google Drive:')
print(f'   MyDrive/smart-learning-assistant/data/raw/1_textbooks/     ← Gonzalez & Woods DIP textbook')
print(f'   MyDrive/smart-learning-assistant/data/raw/2_core_vision/   ← opencv2ref, numpy-user, scipy-ref')
print(f'   MyDrive/smart-learning-assistant/data/raw/3_python_utilities/ ← Matplotlib, Pillow')
print()
print('   Open Drive: https://drive.google.com/drive/my-drive')

# ── 2b. Clone repo (code only) — always pull latest if already cloned ─────
REPO_DIR  = '/content/PixelLab-StudyPal-RAG-DIP'
CODE_ROOT = f'{REPO_DIR}/smart-learning-assistant'

if not os.path.exists(REPO_DIR):
    print('📥 Cloning repository...')
    !git clone --depth 1 https://github.com/Ziadelshazly22/PixelLab-StudyPal-RAG-DIP.git {REPO_DIR}
    print('✅ Repository cloned.')
else:
    print('🔄 Repository already present — pulling latest changes...')
    !git -C {REPO_DIR} pull
    print('✅ Repository updated.')

# Verify app/ is importable
assert os.path.isdir(f'{CODE_ROOT}/app'), \
    f"app/ not found at {CODE_ROOT} — check the repo structure."
print(f'✅ Code root ready.  CODE_ROOT: {CODE_ROOT}')


Mounted at /content/drive
✅ Drive mounted.  DATA_ROOT : /content/drive/MyDrive/smart-learning-assistant

📂 Before running Cell 5, upload your PDFs to Google Drive:
   MyDrive/smart-learning-assistant/data/raw/1_textbooks/     ← Gonzalez & Woods DIP textbook
   MyDrive/smart-learning-assistant/data/raw/2_core_vision/   ← opencv2ref, numpy-user, scipy-ref
   MyDrive/smart-learning-assistant/data/raw/3_python_utilities/ ← Matplotlib, Pillow

   Open Drive: https://drive.google.com/drive/my-drive
📥 Cloning repository...
Cloning into '/content/PixelLab-StudyPal-RAG-DIP'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 35 (delta 0), reused 27 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 24.37 KiB | 6.09 MiB/s, done.
✅ Repository cloned.
✅ Code root ready.  CODE_ROOT: /content/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant


In [3]:
# ── Cell 3: API Key Setup ─────────────────────────────────────────────────
#
#  ┌─────────────────────────────────────────────────────────────────────┐
#  │  NO SECRETS SET UP?  Just run this cell.                           │
#  │  You will be prompted to paste your key — input is hidden.         │
#  │  Get a free key at: https://aistudio.google.com/app/apikey         │
#  └─────────────────────────────────────────────────────────────────────┘
#
#  Optional (avoids the prompt on every session):
#    Colab left sidebar → 🔑 icon → "Add new secret"
#    Name: GOOGLE_API_KEY → paste value → enable "Notebook access" toggle

import os
import getpass

def _get_api_key() -> str:
    # 1. Colab Secrets (if configured in the 🔑 sidebar panel)
    try:
        from google.colab import userdata # type: ignore
        key = userdata.get('GOOGLE_API_KEY')
        if key:
            print('✅ API key loaded from Colab Secrets.')
            return key
    except Exception:
        pass  # Secrets not configured or not in browser UI — continue

    # 2. Environment variable (already set upstream, e.g. from a parent shell)
    key = os.environ.get('GOOGLE_API_KEY', '').strip()
    if key:
        print('✅ API key found in environment variable.')
        return key

    # 3. Interactive prompt — paste your key when asked (input is hidden)
    print('📋 Paste your Google AI Studio API key below.')
    print('   Input is hidden — press Enter when done.')
    print('   (Get a free key at https://aistudio.google.com/app/apikey)\n')
    key = getpass.getpass('GOOGLE_API_KEY:').strip()
    if not key:
        raise ValueError(
            'No API key provided. Get one at https://aistudio.google.com/app/apikey'
        )
    print('✅ API key accepted.')
    return key

os.environ['GOOGLE_API_KEY']     = _get_api_key()
os.environ['EMBEDDING_MODEL']    = 'models/text-embedding-004'
os.environ['CHROMA_PERSIST_DIR'] = f'{DATA_ROOT}/data/chroma_db'

assert os.environ.get('GOOGLE_API_KEY'), 'GOOGLE_API_KEY is empty!'
print('✅ All environment variables configured.')


📋 Paste your Google AI Studio API key below.
   Input is hidden — press Enter when done.
   (Get a free key at https://aistudio.google.com/app/apikey)

✅ API key accepted.
✅ All environment variables configured.


In [4]:
# ── Cell 4: Load Pipeline Module ──────────────────────────────────────────
# Adds CODE_ROOT (the cloned repo) to sys.path so app/ is importable.
# Evicts any stale cached module state before importing.
import sys

# Remove any stale app.* entries from a previous (failed) import attempt
stale = [k for k in sys.modules if k == 'app' or k.startswith('app.')]
for k in stale:
    del sys.modules[k]
if stale:
    print(f'🧹 Cleared {len(stale)} stale module(s): {stale}')

sys.path.insert(0, CODE_ROOT)

try:
    from app.ingestion.pipeline import run_ingestion_pipeline
    print('✅ Pipeline module loaded')
except Exception as _err:
    import traceback
    print('❌ Import failed — real traceback below:')
    traceback.print_exc()
    raise


✅ Pipeline module loaded


In [ ]:
# ── Cell 4b: 🔑 Update API Key (run only if key expired / changed) ────────
#
#  Run this cell ONLY if you see "API key expired" errors in Cell 5.
#  It overwrites the key set in Cell 3 for the current session.
#  Input is hidden — paste your new key and press Enter.
#
#  Get a fresh key at: https://aistudio.google.com/app/apikey

import os, getpass
_new_key = getpass.getpass('Paste new GOOGLE_API_KEY: ').strip()
if not _new_key:
    raise ValueError('No key entered.')
os.environ['GOOGLE_API_KEY'] = _new_key
print('✅ GOOGLE_API_KEY updated for this session.')

✅ GOOGLE_API_KEY updated for this session.
   Now re-run Cell 5 to start ingestion with the new key.


In [ ]:
# ── Cell 5: Run Ingestion ─────────────────────────────────────────────────
# PDFs are read from Drive (DATA_ROOT); results are persisted to Drive too.
import glob

# Pre-flight: confirm PDFs exist before starting
pdf_files = glob.glob(f'{DATA_ROOT}/data/raw/**/*.pdf', recursive=True)
if not pdf_files:
    raise FileNotFoundError(
        f'\n❌  No PDFs found under:\n   {DATA_ROOT}/data/raw/\n\n'
        'Upload your PDF files to Google Drive first (see Cell 2 instructions),\n'
        'then re-run this cell.'
    )

print(f'📄 Found {len(pdf_files)} PDF(s) ready to ingest:')
for f in sorted(pdf_files):
    print(f'   {f}')
print()

stats = run_ingestion_pipeline(
    raw_dir=f'{DATA_ROOT}/data/raw',
    persist_dir=f'{DATA_ROOT}/data/chroma_db',
)

print('\n📊 Ingestion Stats:')
print(f"   Files processed : {stats['processed_files']}")
print(f"   Files skipped   : {stats['skipped_files']}")
print(f"   Total chunks    : {stats['total_chunks']}")
if stats['errors']:
    print(f"   ⚠️  Errors ({len(stats['errors'])})  : {stats['errors']}")


📄 Found 6 PDF(s) ready to ingest:
   /content/drive/MyDrive/smart-learning-assistant/data/raw/1_textbooks/Digital_Image_Processing_Gonzalez_Woods_4th_Ed.pdf
   /content/drive/MyDrive/smart-learning-assistant/data/raw/2_core_vision/numpy-user.pdf
   /content/drive/MyDrive/smart-learning-assistant/data/raw/2_core_vision/opencv2ref.pdf
   /content/drive/MyDrive/smart-learning-assistant/data/raw/2_core_vision/scipy-ref.pdf
   /content/drive/MyDrive/smart-learning-assistant/data/raw/3_python_utilities/Matplotlib.pdf
   /content/drive/MyDrive/smart-learning-assistant/data/raw/3_python_utilities/pillow.pdf



Embedding batches:   0%|          | 0/8 [00:00<?, ?batch/s]ERROR:app.ingestion.pipeline:Failed to store batch after 3 retries — skipping 50 chunks.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_google_genai/embeddings.py", line 432, in embed_documents
    result = self.client.models.embed_content(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 5505, in embed_content
    return self._embed_content(model=model, contents=contents, config=config)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 4505, in _embed_content
    response = self._api_client.request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1401, in request
    response = self._request(http_request, http_options, stream=Fa

In [ ]:
# ── Cell 6: Verify Collection ─────────────────────────────────────────────
# Connects directly to ChromaDB and confirms the expected chunk count.
import chromadb

client = chromadb.PersistentClient(f'{DATA_ROOT}/data/chroma_db')
collection = client.get_collection('dip_knowledge_base')

count = collection.count()
print(f'✅ Collection verified: {count} chunks stored')

sample = collection.peek(3)
print('\nSample metadata:')
for meta in (sample['metadatas'] or []):
    print(f'  • {meta}')


## ⬇️ Next: Download `chroma_db/` to your local machine

Run the cell below to create a zip archive, then download it.
Once downloaded, unzip and place the contents at:

```
smart-learning-assistant/data/chroma_db/
```

> **Note:** The existing local `chroma_db/` folder (if any) should be replaced entirely.

In [ ]:
# ── Cell 7: Zip & Download chroma_db/ ────────────────────────────────────
import shutil
from google.colab import files # type: ignore

archive_path = '/content/chroma_db_export'
shutil.make_archive(archive_path, 'zip', f'{DATA_ROOT}/data', 'chroma_db')
print(f'✅ Archive created: {archive_path}.zip')

files.download(f'{archive_path}.zip')
print('✅ Download started. Unzip into your local smart-learning-assistant/data/ folder.')
